# Chapter 8 — RAG Generation
## Topic 7: Query Rewriting and HyDE (Hypothetical Document Embeddings)

**Run cells in order.**

- Module 1: Setup — Hinglish query + mock HyDE-generation client
- Module 2: Simple query expansion (Hinglish->English term mapping)
- Module 3: HyDE — generate hypothetical answer, embed it, retrieve with it
- Module 4: Head-to-head — raw query vs. query expansion vs. HyDE

**Install:** `pip install sentence-transformers numpy`


## Module 1: Setup — Hinglish Query + Mock HyDE Client

**Requires:** nothing standalone

In [ ]:
"""
MODULE 1: Setup

MockClient simulates the HyDE-generation LLM call offline. Replace with
a real anthropic.Anthropic() client + claude-haiku-4-5-20251001 in
production -- the function signature (Module 3) is identical either way.
"""

KNOWLEDGE_BASE = [
    {"id": 0, "text": "Premature closure of a Fixed Deposit is permitted subject to a penalty of 1 percent on the applicable interest rate, payable at the time of closure."},
    {"id": 1, "text": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures."},
    {"id": 2, "text": "The Fixed Deposit interest rate for 24 months is 7.1 percent per annum."},
]

HINGLISH_QUERY = "FD band karna hai penalty kitni hogi"  # "I want to close my FD, what's the penalty?"

HINGLISH_TO_ENGLISH = {
    "fd": "fixed deposit",
    "band": "close closure",
    "karna": "",
    "hai": "",
    "penalty": "penalty",
    "kitni": "how much",
    "hogi": "",
}


class MockHydeClient:
    """Simulates the HyDE-generation LLM call offline. A real implementation
    calls client.messages.create() with claude-haiku-4-5-20251001."""

    class MockResponse:
        def __init__(self, text):
            self.content = [type("Block", (), {"text": text})()]

    class Messages:
        def create(self, model, max_tokens, messages):
            # Simulates a plausible, FORMAL-ENGLISH hypothetical answer --
            # this is the exact mechanism that bridges Hinglish query
            # register to the formal-English knowledge base's register.
            hypothetical = (
                "Premature closure of a Fixed Deposit typically incurs a penalty "
                "of around 1 percent on the applicable interest rate, and the "
                "account is closed within a few working days of the request."
            )
            return MockHydeClient.MockResponse(hypothetical)

    def __init__(self):
        self.messages = MockHydeClient.Messages()


mock_client = MockHydeClient()
MODEL_ID = "claude-haiku-4-5-20251001"

print(f"Hinglish query: \'{HINGLISH_QUERY}\'")
print("(Meaning: \'I want to close my FD, what is the penalty?\')")
print("\nModule 1 complete. Run Module 2.")


## Module 2: Simple Query Expansion (Hinglish -> English)

**Requires:** Module 1

In [ ]:
"""
MODULE 2: Simple Query Expansion

The CHEAPER alternative to HyDE, directly targeting this project's
specifically diagnosed Hinglish vocabulary mismatch problem
(Chapter 7 Topic 1). No LLM call needed -- just a dictionary lookup.
"""

def simple_query_expansion(query: str, hinglish_map: dict) -> str:
    words = query.lower().split()
    expanded = list(words)
    for word in words:
        mapping = hinglish_map.get(word, "")
        for term in mapping.split():
            if term and term not in expanded:
                expanded.append(term)
    return " ".join(expanded)


expanded_query = simple_query_expansion(HINGLISH_QUERY, HINGLISH_TO_ENGLISH)
print(f"Original:  \'{HINGLISH_QUERY}\'")
print(f"Expanded:  \'{expanded_query}\'")

print("\nModule 2 complete. Run Module 3.")


## Module 3: HyDE — Generate, Embed, Retrieve

**Requires:** Module 1

In [ ]:
"""
MODULE 3: HyDE

Generates a hypothetical answer (discarded after use), embeds IT instead
of the original query, retrieves using that embedding.
"""

import numpy as np
from sentence_transformers import SentenceTransformer

print("Loading embedding model (may take a moment on first run)...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

kb_texts = [doc["text"] for doc in KNOWLEDGE_BASE]
kb_embeddings = embed_model.encode(kb_texts, normalize_embeddings=True, show_progress_bar=False)
for i, doc in enumerate(KNOWLEDGE_BASE):
    doc["embedding"] = kb_embeddings[i]


def hyde_rewrite_and_embed(query: str, client, model_id: str, embed_model) -> tuple:
    """Returns (hyde_embedding, hypothetical_answer_text_for_inspection_only).
    The hypothetical answer text is NEVER shown to a customer or cited --
    it exists purely to produce a better embedding for retrieval."""
    hyde_prompt = f"""Write a brief, plausible-sounding answer to this customer question
about Fixed Deposits, as if you were an FD policy document.

Question: {query}

Hypothetical answer:"""

    response = client.messages.create(
        model=model_id, max_tokens=150,
        messages=[{"role": "user", "content": hyde_prompt}],
    )
    hypothetical_answer = response.content[0].text.strip()
    hyde_embedding = embed_model.encode(hypothetical_answer, normalize_embeddings=True, show_progress_bar=False)
    return hyde_embedding, hypothetical_answer


def dense_retrieve_with_vector(query_vec, top_k: int = 3) -> list:
    scored = [(doc, float(np.dot(query_vec, doc["embedding"]))) for doc in KNOWLEDGE_BASE]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


hyde_vec, hypothetical_text = hyde_rewrite_and_embed(HINGLISH_QUERY, mock_client, MODEL_ID, embed_model)

print(f"Hypothetical answer generated (INTERNAL USE ONLY, never shown to customer):")
print(f"  \'{hypothetical_text}\'\n")

hyde_results = dense_retrieve_with_vector(hyde_vec)
print("HyDE-based retrieval results:")
for doc, score in hyde_results:
    print(f"  score={score:.4f} | {doc['text'][:60]}...")

print("\nModule 3 complete. Run Module 4.")


## Module 4: Head-to-Head — Raw vs. Query Expansion vs. HyDE

**Requires:** Module 1, Module 2, Module 3

In [ ]:
"""
MODULE 4: Head-to-Head Comparison

Runs all three approaches on the SAME Hinglish query and compares top-1
scores -- the evidence needed to decide which is actually worth the cost
for THIS project's specific, diagnosed problem.
"""

def dense_retrieve_raw(query: str, top_k: int = 3) -> list:
    q_vec = embed_model.encode(query, normalize_embeddings=True, show_progress_bar=False)
    return dense_retrieve_with_vector(q_vec, top_k)


print("=" * 70)
print(f"Query: \'{HINGLISH_QUERY}\'")
print("=" * 70)

raw_results = dense_retrieve_raw(HINGLISH_QUERY)
expansion_results = dense_retrieve_raw(expanded_query)
# hyde_results already computed in Module 3

print("\n1. RAW query (no rewriting):")
for doc, score in raw_results:
    print(f"   score={score:.4f} | {doc['text'][:55]}...")

print("\n2. QUERY EXPANSION (cheap, targeted, no LLM call):")
for doc, score in expansion_results:
    print(f"   score={score:.4f} | {doc['text'][:55]}...")

print("\n3. HyDE (LLM call required, more general-purpose):")
for doc, score in hyde_results:
    print(f"   score={score:.4f} | {doc['text'][:55]}...")

print(f"\n{'Method':<20} | {'Top-1 score':<12} | {'Extra LLM call?':<16}")
print("-" * 55)
print(f"{'Raw query':<20} | {raw_results[0][1]:<12.4f} | {'No':<16}")
print(f"{'Query expansion':<20} | {expansion_results[0][1]:<12.4f} | {'No':<16}")
print(f"{'HyDE':<20} | {hyde_results[0][1]:<12.4f} | {'Yes (+cost/latency)':<16}")

print("""
Interpretation: compare the top-1 score improvement of Query Expansion
(free) against HyDE (costs an extra LLM call) over the Raw baseline.
If Query Expansion alone captures most of the improvement, per this
project's theory recommendation, it's the better default given its
much lower cost -- reserve HyDE for cases where evaluation (Chapter 7
Topic 9's methodology) shows it adds meaningful further improvement
beyond what the free fix already provides.
""")

print("=" * 70)
print("CHAPTER 8 TOPIC 7 SUMMARY")
print("=" * 70)
print("""
Query expansion: cheap, dictionary-based, directly targets this project's
  diagnosed Hinglish vocabulary mismatch (Chapter 7 Topic 1).
HyDE: generates a hypothetical (ungrounded, DISCARDED-after-use) answer,
  embeds THAT instead of the raw query -- bridges register/style mismatch
  more generally, at the cost of an extra LLM call per query.
HyDE helps DENSE retrieval specifically, not BM25 (exact-term matching
  doesn't benefit from a paraphrased embedding the same way).
For this project: measure query expansion first (free) before adopting
  HyDE's added per-query cost, using Chapter 7 Topic 9's evaluation
  methodology -- don't assume HyDE's general reputation transfers
  automatically to this project's SPECIFIC, already-diagnosed problem.

Next: Topic 8 -- RAG vs. Fine-Tuning
""")
